# Stratified Sampling & Skill Extraction from Job Descriptions

This notebook handles large-scale job description data (20M+ records) by:
1. **Stratified sampling**: Sample N records per quarter × onet_code × rcid
2. **Skill extraction**: Extract KSAO skills using semantic matching
3. **Analysis**: Temporal trends and skill evolution

**Strategy:**
- Process 200-300 parquet files without loading all into memory
- Use DuckDB for efficient sampling across files
- Extract skills from sampled data only

**Sampling rationale:**
- Original: 20M+ records → Too large to process
- Sampled: ~100K-500K records → Manageable & representative

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
from glob import glob
from tqdm import tqdm
import json
import os
import duckdb
from datetime import datetime

from skillner.jd_skill_extractor import JobDescriptionSkillExtractor

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

print("✓ Imports successful")

## Configuration

**EDIT THESE SETTINGS:**

In [ ]:
# =====================================================================
# EDIT THESE PATHS AND PARAMETERS
# =====================================================================

# Input folder with multiple parquet files
INPUT_FOLDER = '../JD'  # Folder containing 200-300 parquet files

# Knowledge base
KB_PATH = '../.skillner-kb/MERGED_EN.pkl'  # Or ONET_EN.pkl

# Output paths
SAMPLED_DATA_PATH = '../data/jd_sampled_stratified.parquet'  # Sampled data
OUTPUT_PATH = '../data/jd_extracted_skills_sampled.parquet'  # Final results
STATS_PATH = '../data/jd_stats_sampled.json'

# Column names in your data
JD_TEXT_COLUMN = 'job_description'
ONET_CODE_COLUMN = 'onet_code'
RCID_COLUMN = 'rcid'  # Company/recruiter ID
DATE_COLUMN = 'post_date'  # Format: '2023-01-19'

# =====================================================================
# SAMPLING PARAMETERS - ADJUST BASED ON YOUR NEEDS
# =====================================================================

# Number of records to sample per quarter × onet_code × rcid group
SAMPLE_SIZE_PER_GROUP = 20  # Options: 5 (quick), 10, 20 (recommended), 50 (thorough)

# Optional: Limit to specific date range
DATE_START = None  # e.g., '2020-01-01' or None for all
DATE_END = None    # e.g., '2023-12-31' or None for all

# Optional: Limit to specific ONET codes (for testing)
ONET_CODES_FILTER = None  # e.g., ['15-1252.00', '11-1021.00'] or None for all

# =====================================================================
# EXTRACTION PARAMETERS
# =====================================================================

SIMILARITY_THRESHOLD = 0.6  # Lower = more matches, higher = stricter
MAX_WINDOW_SIZE = 5  # Maximum words in a skill phrase

# =====================================================================

print(f"Configuration:")
print(f"  Input folder: {INPUT_FOLDER}")
print(f"  KB: {KB_PATH}")
print(f"  Sampled data output: {SAMPLED_DATA_PATH}")
print(f"  Final results output: {OUTPUT_PATH}")
print(f"\nSampling strategy:")
print(f"  Records per group (quarter × onet_code × rcid): {SAMPLE_SIZE_PER_GROUP}")
print(f"  Date range: {DATE_START or 'All'} to {DATE_END or 'All'}")
print(f"  ONET filter: {ONET_CODES_FILTER or 'All codes'}")
print(f"\nExtraction parameters:")
print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"  Max window size: {MAX_WINDOW_SIZE}")

## Step 1: Discover Input Files

In [ ]:
# Find all parquet files
input_files = sorted(glob(f"{INPUT_FOLDER}/*.parquet"))

print(f"Found {len(input_files)} parquet files in {INPUT_FOLDER}")

if len(input_files) == 0:
    print(f"\n⚠️ WARNING: No parquet files found in {INPUT_FOLDER}")
    print("Please check the INPUT_FOLDER path.")
else:
    print(f"\nFirst 5 files:")
    for f in input_files[:5]:
        file_size = os.path.getsize(f) / (1024 * 1024)  # MB
        print(f"  {os.path.basename(f):40s} ({file_size:.1f} MB)")
    
    if len(input_files) > 5:
        print(f"  ... and {len(input_files) - 5} more files")
    
    # Calculate total size
    total_size_gb = sum(os.path.getsize(f) for f in input_files) / (1024**3)
    print(f"\nTotal size: {total_size_gb:.2f} GB")

## Step 2: Sample Data Overview

Check one file to understand data structure:

In [ ]:
# Safe parquet reader to avoid PyArrow extension type conflicts
import pyarrow.parquet as pq

def read_parquet_safe(file_path):
    """Safely read parquet files avoiding PyArrow extension type conflicts."""
    try:
        table = pq.read_table(file_path)
        return table.to_pandas()
    except Exception:
        table = pq.read_table(file_path, use_pandas_metadata=False)
        return table.to_pandas()

# Load first file as sample
sample_file = input_files[0]
print(f"Loading sample file: {os.path.basename(sample_file)}")

df_sample = read_parquet_safe(sample_file)

print(f"\n✓ Loaded {len(df_sample):,} rows")
print(f"\nColumns: {list(df_sample.columns)}")
print(f"\nRequired columns check:")
print(f"  {JD_TEXT_COLUMN}: {'✓' if JD_TEXT_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
print(f"  {ONET_CODE_COLUMN}: {'✓' if ONET_CODE_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
print(f"  {RCID_COLUMN}: {'✓' if RCID_COLUMN in df_sample.columns else '✗ NOT FOUND'}")
print(f"  {DATE_COLUMN}: {'✓' if DATE_COLUMN in df_sample.columns else '✗ NOT FOUND'}")

print(f"\nSample data:")
display(df_sample[[DATE_COLUMN, ONET_CODE_COLUMN, RCID_COLUMN]].head(10))

# Check date format
print(f"\nDate examples:")
print(df_sample[DATE_COLUMN].head(5).tolist())

## Step 3: Stratified Sampling with DuckDB

**This step performs stratified sampling across all files without loading everything into memory.**

Strategy:
- Use DuckDB to query all parquet files
- Extract quarter from post_date
- Sample N records per quarter × onet_code × rcid
- Save sampled data

In [ ]:
print("="*70)
print("Stratified Sampling with DuckDB")
print("="*70)
print("This may take 5-15 minutes depending on data size...\n")

# Create DuckDB connection
conn = duckdb.connect()

# Build file pattern
files_pattern = f"{INPUT_FOLDER}/*.parquet"

# Build date filter
date_filter = ""
if DATE_START and DATE_END:
    date_filter = f"AND CAST({DATE_COLUMN} AS DATE) BETWEEN DATE '{DATE_START}' AND DATE '{DATE_END}'"
elif DATE_START:
    date_filter = f"AND CAST({DATE_COLUMN} AS DATE) >= DATE '{DATE_START}'"
elif DATE_END:
    date_filter = f"AND CAST({DATE_COLUMN} AS DATE) <= DATE '{DATE_END}'"

# Build ONET filter
onet_filter = ""
if ONET_CODES_FILTER:
    onet_list = "', '".join(ONET_CODES_FILTER)
    onet_filter = f"AND {ONET_CODE_COLUMN} IN ('{onet_list}')"

# Stratified sampling query using QUALIFY (DuckDB feature)
sampling_query = f"""
SELECT
    *,
    CAST(YEAR(CAST({DATE_COLUMN} AS DATE)) AS VARCHAR) || 'Q' ||
    CAST(QUARTER(CAST({DATE_COLUMN} AS DATE)) AS VARCHAR) AS quarter
FROM read_parquet('{files_pattern}')
WHERE {JD_TEXT_COLUMN} IS NOT NULL
    AND {ONET_CODE_COLUMN} IS NOT NULL
    AND {RCID_COLUMN} IS NOT NULL
    {date_filter}
    {onet_filter}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY 
        CAST(YEAR(CAST({DATE_COLUMN} AS DATE)) AS VARCHAR) || 'Q' ||
        CAST(QUARTER(CAST({DATE_COLUMN} AS DATE)) AS VARCHAR),
        {ONET_CODE_COLUMN}, 
        {RCID_COLUMN}
    ORDER BY {DATE_COLUMN}
) <= {SAMPLE_SIZE_PER_GROUP}
ORDER BY quarter, {ONET_CODE_COLUMN}, {RCID_COLUMN}, {DATE_COLUMN}
"""

print("Executing sampling query...")
print(f"Sampling {SAMPLE_SIZE_PER_GROUP} records per quarter × onet_code × rcid")
print("Writing directly to parquet (no pandas conversion)...\n")

try:
    # Write directly to parquet using DuckDB COPY command
    # This avoids loading all data into pandas/memory
    copy_query = f"""
    COPY (
        {sampling_query}
    ) TO '{SAMPLED_DATA_PATH}' (FORMAT PARQUET)
    """
    
    conn.execute(copy_query)
    
    print(f"✓ Sampling complete! Data written to {SAMPLED_DATA_PATH}")
    
    # Get statistics using DuckDB (no pandas loading)
    print(f"\nGathering statistics...")
    
    # Total records
    total_count = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{SAMPLED_DATA_PATH}')").fetchone()[0]
    
    # Unique quarters
    unique_quarters = conn.execute(f"SELECT COUNT(DISTINCT quarter) FROM read_parquet('{SAMPLED_DATA_PATH}')").fetchone()[0]
    
    # Unique ONET codes
    unique_onet = conn.execute(f"SELECT COUNT(DISTINCT {ONET_CODE_COLUMN}) FROM read_parquet('{SAMPLED_DATA_PATH}')").fetchone()[0]
    
    # Unique RCIDs
    unique_rcids = conn.execute(f"SELECT COUNT(DISTINCT {RCID_COLUMN}) FROM read_parquet('{SAMPLED_DATA_PATH}')").fetchone()[0]
    
    # Group counts
    group_count = conn.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT quarter, {ONET_CODE_COLUMN}, {RCID_COLUMN}
            FROM read_parquet('{SAMPLED_DATA_PATH}')
            GROUP BY quarter, {ONET_CODE_COLUMN}, {RCID_COLUMN}
        )
    """).fetchone()[0]
    
    print(f"\nSampled data summary:")
    print(f"  Total records sampled: {total_count:,}")
    print(f"  Unique quarters: {unique_quarters}")
    print(f"  Unique ONET codes: {unique_onet}")
    print(f"  Unique RCIDs: {unique_rcids:,}")
    print(f"  Unique groups (quarter × onet × rcid): {group_count:,}")
    
    # Show distribution by quarter
    print(f"\nQuarters:")
    quarter_dist = conn.execute(f"""
        SELECT quarter, COUNT(*) as count
        FROM read_parquet('{SAMPLED_DATA_PATH}')
        GROUP BY quarter
        ORDER BY quarter
        LIMIT 10
    """).fetchall()
    
    for q, count in quarter_dist:
        print(f"  {q}: {count:,} records")
    
    if unique_quarters > 10:
        print(f"  ... and {unique_quarters - 10} more quarters")
    
    print(f"\n✓ Sampled data saved to: {SAMPLED_DATA_PATH}")
    print(f"  File size: {os.path.getsize(SAMPLED_DATA_PATH) / (1024**2):.1f} MB")
    
except Exception as e:
    print(f"\n✗ Error during sampling: {e}")
    print("\nTroubleshooting:")
    print("  1. Check that column names are correct")
    print("  2. Verify date format in your data")
    print("  3. Check if all required columns exist")
    print(f"  4. Check write permissions for: {SAMPLED_DATA_PATH}")
    raise
finally:
    conn.close()

## Step 4: Explore Sampled Data

In [ ]:
# Load sampled data
df_sampled = read_parquet_safe(SAMPLED_DATA_PATH)

print(f"Sampled data loaded: {len(df_sampled):,} records")
print(f"Memory usage: {df_sampled.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

# Check null values
print(f"\nNull job descriptions: {df_sampled[JD_TEXT_COLUMN].isna().sum():,} ({df_sampled[JD_TEXT_COLUMN].isna().sum() / len(df_sampled) * 100:.1f}%)")

# Show distribution
print(f"\nTop 10 ONET codes:")
for onet, count in df_sampled[ONET_CODE_COLUMN].value_counts().head(10).items():
    print(f"  {onet}: {count:,} records")

print(f"\nSample records:")
display(df_sampled[['quarter', ONET_CODE_COLUMN, RCID_COLUMN, JD_TEXT_COLUMN]].head(10))

## Step 5: Estimate Processing Time

In [ ]:
# Estimate processing time
num_jds = len(df_sampled[df_sampled[JD_TEXT_COLUMN].notna()])

time_per_jd_min = 2  # seconds
time_per_jd_max = 5  # seconds

total_time_min_hours = (num_jds * time_per_jd_min) / 3600
total_time_max_hours = (num_jds * time_per_jd_max) / 3600

print("="*70)
print("Processing Time Estimate")
print("="*70)
print(f"Job descriptions to process: {num_jds:,}")
print(f"Estimated time per JD: {time_per_jd_min}-{time_per_jd_max} seconds")
print(f"\nTotal estimated time:")
print(f"  Minimum: {total_time_min_hours:.1f} hours")
print(f"  Maximum: {total_time_max_hours:.1f} hours")
print(f"  Average: {(total_time_min_hours + total_time_max_hours) / 2:.1f} hours")
print("\nNote: You can stop and the progress will be saved.")

## Step 6: Initialize Skill Extractor

In [ ]:
print("Initializing skill extractor...")
print("(This may take 1-2 minutes to load the semantic model)\n")

extractor = JobDescriptionSkillExtractor(
    kb_path=KB_PATH,
    model_name='all-MiniLM-L6-v2',
    similarity_threshold=SIMILARITY_THRESHOLD,
    max_window_size=MAX_WINDOW_SIZE
)

print("\n✓ Extractor ready!")

## Step 7: Test on Single Example

In [ ]:
# Get first non-null job description
test_jd = df_sampled[df_sampled[JD_TEXT_COLUMN].notna()][JD_TEXT_COLUMN].iloc[0]

print("Test Job Description:")
print("=" * 70)
print(test_jd[:500] + "..." if len(test_jd) > 500 else test_jd)
print("=" * 70)

# Extract skills
print("\nExtracting skills...\n")
result = extractor.extract_skills(test_jd, return_details=True)

print(f"✓ Found {result['num_skills']} unique skills")
print(f"\nSkills by category:")
for section, skills in result['by_section'].items():
    print(f"  [{section}]: {len(skills)} skills")
    for skill in skills[:3]:
        print(f"    - {skill}")
    if len(skills) > 3:
        print(f"    ... and {len(skills) - 3} more")

print(f"\nAll extracted skills:")
for i, skill in enumerate(sorted(result['skills']), 1):
    print(f"{i:2d}. {skill}")

## Step 8: Extract Skills from All Sampled Data

**This will process all sampled job descriptions.**

In [ ]:
print("="*70)
print("Extracting Skills from Sampled Data")
print("="*70)
print(f"Processing {len(df_sampled):,} sampled job descriptions...\n")

# Get list of job descriptions
jd_list = df_sampled[JD_TEXT_COLUMN].tolist()

# Extract skills with progress bar
results = extractor.extract_skills_batch(jd_list, show_progress=True)

print(f"\n✓ Extraction complete!")

## Step 9: Combine Results with Original Data

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame([
    {
        'skills': r['skills'],
        'num_skills': r['num_skills'],
        'by_section': r['by_section']
    }
    for r in results
])

# Combine with original data
df_final = pd.concat([df_sampled.reset_index(drop=True), results_df], axis=1)

print(f"Combined DataFrame shape: {df_final.shape}")
print(f"\nColumns: {list(df_final.columns)}")
print(f"\nSample:")
display(df_final[['quarter', ONET_CODE_COLUMN, 'num_skills']].head(10))

## Step 10: Basic Statistics

In [ ]:
# Get statistics
stats = extractor.get_statistics(results)

print("="*70)
print("Extraction Statistics")
print("="*70)

print(f"\nTotal job descriptions: {stats['total_jds']:,}")
print(f"Total unique skills found: {stats['unique_skills_total']:,}")

print(f"\nSkills per job description:")
print(f"  Mean: {stats['skills_per_jd']['mean']:.1f}")
print(f"  Median: {stats['skills_per_jd']['median']:.1f}")
print(f"  Min: {stats['skills_per_jd']['min']:.0f}")
print(f"  Max: {stats['skills_per_jd']['max']:.0f}")
print(f"  Std: {stats['skills_per_jd']['std']:.1f}")

print(f"\nSkills by section:")
for section, section_stats in stats['by_section'].items():
    print(f"  [{section}]")
    print(f"    Mean: {section_stats['mean']:.1f}")
    print(f"    Median: {section_stats['median']:.1f}")

print(f"\nTop 20 most common skills:")
for i, (skill, count) in enumerate(stats['top_10_skills'][:20] if 'top_10_skills' in stats else [], 1):
    pct = (count / stats['total_jds']) * 100
    print(f"{i:2d}. {skill:45s} ({count:6,} times, {pct:5.1f}%)")

## Step 11: Visualizations

In [ ]:
# Distribution of skills per JD
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_final['num_skills'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Skills per Job Description')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Skills per JD')
axes[0].axvline(stats['skills_per_jd']['mean'], color='red', linestyle='--', 
               label=f"Mean: {stats['skills_per_jd']['mean']:.1f}")
axes[0].legend()

# Box plot by ONET code (top 10)
top_onet = df_final[ONET_CODE_COLUMN].value_counts().head(10).index
df_top = df_final[df_final[ONET_CODE_COLUMN].isin(top_onet)]
df_top.boxplot(column='num_skills', by=ONET_CODE_COLUMN, ax=axes[1], rot=45)
axes[1].set_xlabel('ONET Code')
axes[1].set_ylabel('Number of Skills')
axes[1].set_title('Skills per JD by ONET Code (Top 10)')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Skills by KSAO category
section_counts = {}
for result in results:
    for section, skills in result['by_section'].items():
        section_counts[section] = section_counts.get(section, 0) + len(skills)

plt.figure(figsize=(12, 6))
sections = list(section_counts.keys())
counts = list(section_counts.values())
plt.bar(sections, counts, edgecolor='black', alpha=0.7)
plt.xlabel('KSAO Section')
plt.ylabel('Total Skills Extracted')
plt.title('Skills Extracted by KSAO Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nSkills by section:")
for section, count in sorted(section_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {section:30s}: {count:6,} skills")

## Step 12: Time Series Analysis

In [ ]:
# Average skills per quarter
quarterly_stats = df_final.groupby('quarter')['num_skills'].agg(['mean', 'median', 'count'])

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Skills over time
axes[0].plot(quarterly_stats.index.astype(str), quarterly_stats['mean'], 
            marker='o', label='Mean', linewidth=2)
axes[0].plot(quarterly_stats.index.astype(str), quarterly_stats['median'], 
            marker='s', label='Median', linewidth=2)
axes[0].set_xlabel('Quarter')
axes[0].set_ylabel('Skills per JD')
axes[0].set_title('Average Skills per Job Description Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Number of JDs per quarter
axes[1].bar(quarterly_stats.index.astype(str), quarterly_stats['count'], 
           edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Number of Job Descriptions (Sampled)')
axes[1].set_title('Sampled Job Descriptions per Quarter')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\nQuarterly statistics:")
display(quarterly_stats)

## Step 13: KSAO Category Trends Over Time

In [ ]:
# Calculate skills by section per quarter
section_by_quarter = {}

for idx, row in df_final.iterrows():
    quarter = row['quarter']
    for section, skills in row['by_section'].items():
        if quarter not in section_by_quarter:
            section_by_quarter[quarter] = {}
        if section not in section_by_quarter[quarter]:
            section_by_quarter[quarter][section] = 0
        section_by_quarter[quarter][section] += len(skills)

# Convert to DataFrame
section_trends = pd.DataFrame(section_by_quarter).T.fillna(0)
section_trends = section_trends.sort_index()

# Normalize by number of JDs per quarter
jd_counts = df_final.groupby('quarter').size()
section_trends_normalized = section_trends.div(jd_counts, axis=0)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

for section in section_trends_normalized.columns:
    ax.plot(section_trends_normalized.index.astype(str), 
           section_trends_normalized[section], 
           marker='o', label=section, linewidth=2)

ax.set_xlabel('Quarter')
ax.set_ylabel('Average Skills per JD')
ax.set_title('KSAO Category Trends Over Time (Normalized)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nKSAO trends (average per JD):")
display(section_trends_normalized.head(10))

## Step 14: Top Skills by ONET Code

In [ ]:
# Analyze top 5 ONET codes
top_5_onet = df_final[ONET_CODE_COLUMN].value_counts().head(5).index

print("="*70)
print("Top Skills by ONET Code (Top 5 Occupations)")
print("="*70)

for onet in top_5_onet:
    df_onet = df_final[df_final[ONET_CODE_COLUMN] == onet]
    
    # Collect all skills for this ONET
    onet_skills = []
    for skills_list in df_onet['skills']:
        onet_skills.extend(skills_list)
    
    from collections import Counter
    skill_freq = Counter(onet_skills)
    
    print(f"\n{onet} (n={len(df_onet):,}):")
    for i, (skill, count) in enumerate(skill_freq.most_common(10), 1):
        pct = (count / len(df_onet)) * 100
        print(f"  {i:2d}. {skill:40s} ({count:4d} times, {pct:5.1f}%)")

## Step 15: Save Results

In [ ]:
# Save final results
print(f"Saving final results to {OUTPUT_PATH}...")
df_final.to_parquet(OUTPUT_PATH, index=False)
print(f"✓ Saved {len(df_final):,} records")

# Save statistics as JSON
stats_summary = {
    'sampling': {
        'sample_size_per_group': SAMPLE_SIZE_PER_GROUP,
        'total_sampled': len(df_final),
        'unique_quarters': int(df_final['quarter'].nunique()),
        'unique_onet_codes': int(df_final[ONET_CODE_COLUMN].nunique()),
        'unique_rcids': int(df_final[RCID_COLUMN].nunique()),
        'date_range': {
            'start': DATE_START,
            'end': DATE_END
        }
    },
    'extraction': {
        'total_jds': stats['total_jds'],
        'unique_skills_total': stats['unique_skills_total'],
        'skills_per_jd': stats['skills_per_jd'],
        'by_section': stats['by_section'],
        'top_20_skills': [[skill, int(count)] for skill, count in stats.get('top_10_skills', [])[:20]]
    },
    'parameters': {
        'similarity_threshold': SIMILARITY_THRESHOLD,
        'max_window_size': MAX_WINDOW_SIZE,
        'kb_path': KB_PATH
    }
}

with open(STATS_PATH, 'w') as f:
    json.dump(stats_summary, f, indent=2)
print(f"✓ Saved statistics to {STATS_PATH}")

print("\n" + "="*70)
print("ALL PROCESSING COMPLETE")
print("="*70)
print(f"Sampled data: {SAMPLED_DATA_PATH}")
print(f"Results: {OUTPUT_PATH}")
print(f"Statistics: {STATS_PATH}")
print(f"\nSummary:")
print(f"  Sampled: {len(df_final):,} job descriptions")
print(f"  Extracted: {stats['unique_skills_total']:,} unique skills")
print(f"  Quarters: {df_final['quarter'].nunique()}")
print(f"  ONET codes: {df_final[ONET_CODE_COLUMN].nunique()}")

## Summary

This notebook:
1. ✅ Performed stratified sampling on 20M+ records
2. ✅ Sampled N records per quarter × onet_code × rcid
3. ✅ Extracted KSAO skills using semantic matching
4. ✅ Analyzed temporal trends
5. ✅ Identified top skills by occupation

**Next steps:**
- Analyze specific occupations or time periods
- Identify emerging skills (increasing frequency)
- Compare skill requirements across occupations
- Study KSAO category evolution